***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [8. 校准](8_0_introduction.ipynb)
    * 上一节： [8.3 第二代校准（2GC）](8_3_2gc.ipynb)
    * 下一节： [8.5 延伸阅读与参考文献](8_5_further_reading_and_references.ipynb)

***


## 8.4 第三代校准（3GC）：方向依赖自校准

第一代校准和第二代校准的共同近似，是每面天线在整个目标场内可以用一个方向无关 Jones 项描述。这个近似在小视场、窄带和方向依赖效应较弱时很有效；但在宽场、低频、高动态范围或精密极化观测中，它往往不再成立。主波束、指向误差、电离层相位屏、对流层水汽、离轴极化泄漏和天线间波束差异，都会使同一台天线对不同天空方向具有不同响应。

第三代校准（3GC）的任务，就是把这些方向依赖效应纳入测量方程，使残差不再随天空位置系统性增长。它不是某一个特定算法的名称，而是一组共同思想：从“每台天线一个增益”扩展到“每台天线在不同方向、时间、频率和极化上都有可建模响应”。


### 8.4.1 方向依赖项如何进入 RIME

在方向无关标量近似下，可见度常写成 $V_{pq}=g_pX_{pq}g_q^*$。若天空亮度由多个方向组成，并且天线响应依赖方向，则更合适的写法是

$$
V_{pq}=\sum_s E_p(\boldsymbol{s})\,g_p\,I_s\,
\exp[-2\pi i\boldsymbol{u}_{pq}\cdot\boldsymbol{s}]\,
g_q^*E_q^*(\boldsymbol{s}).
$$

这里 $E_p(\boldsymbol{s})$ 是天线 $p$ 在方向 $\boldsymbol{s}$ 上的方向相关响应。若 $E_p$ 对所有方向都相同，它可以并入 $g_p$，问题退化回 2GC；若不同方向的 $E_p$ 明显不同，就不存在一个全场统一增益能够同时校正所有源。求解器若仍使用方向无关模型，通常会优先照顾最亮方向，并在其他方向留下结构化残差。


### 8.4.2 主波束和指向误差的直观例子

主波束是最容易理解的方向依赖项。对视场中心附近的源，小的指向偏差可能只带来微弱振幅变化；但对离轴源，同样的指向偏差会落在波束斜率较大的位置，从而被放大成显著通量误差。不同天线的指向偏差若不相同，每条基线还会看到不同的乘积响应。

这类误差随时间也会变化。天线结构形变、风载、温度和跟踪误差会让离轴源的表观增益在观测过程中漂移。因此，离轴残差常常表现为围绕亮源的时间相关条纹、拉伸或负碗，而不是全场均匀的噪声升高。


![主波束和指向误差造成方向依赖增益](figures/calibration_3gc_beam_pointing.png)

**图 8.4.1** 不同天线的微小指向偏差会让离轴源看到不同主波束增益。右图显示同一天线对不同离轴源的响应还会随时间变化。


### 8.4.3 方向无关模型为什么无法消除方向依赖残差

下面的示意数据由三个点源组成，每台天线具有轻微不同的主波束指向误差。若模型忽略主波束差异，只把三个源的理想可见度相加，就得到方向无关（DI）模型；若模型显式把每个方向上的波束乘积写入预测可见度，就得到方向相关（DD）模型。

二者的差别不只是残差大小不同，而是残差结构不同。DI 模型留下的残差集中在离轴源方向附近，因为这些源的真实表观通量和模型通量不一致；DD 模型把这种方向依赖响应放回测量方程后，结构化残差才从根本上消失。


![方向无关模型与方向相关模型的残差对比](figures/calibration_3gc_di_vs_dd_residual.png)

**图 8.4.2** 同一组方向依赖数据分别用 DI 模型和 DD 模型预测后的残差图。污染本质上依赖方向时，继续调方向无关增益只能隐藏问题，不能消除离轴结构化残差。


### 8.4.4 3GC 的几条主要路线

真实软件中的 3GC 通常有几类路线。物理模型法把主波束、电离层屏或指向误差参数化后直接写入 RIME，再联合求解或在成像中传播。差分增益和 peeling 针对少数亮离轴源单独建立方向相关校正，常用于压制支配动态范围的强源残差。Faceting 把大视场分成若干小块，在每个方向块内近似响应相同。A-projection 和 AW-projection 则把主波束和宽场效应放入成像卷积核，使方向依赖响应在 gridding/degridding 过程中被处理。

这些路线并不互相排斥。实际处理常采用混合策略：先用物理模型吸收大尺度、可预测的方向依赖效应，再对最亮或最麻烦的方向做局部修正。方法选择取决于误差物理、亮源分布、频段、计算预算和科学目标。


![3GC 常见算法路线](figures/calibration_3gc_method_map.png)

**图 8.4.3** 3GC 的几类常见路线都围绕方向相关 Jones 项展开，只是把方向依赖响应放入求解器、局部方向、图像分块或成像卷积核的方式不同。


### 8.4.5 计算代价与正则化

进入 3GC 后，困难不只是未知数更多，而是未知数的结构更复杂。若有 $N_{\rm ant}$ 面天线、$N_{\rm dir}$ 个求解方向、$N_t$ 个时间解、$N_\nu$ 个频率解，并且每个复增益有振幅和相位两个实自由度，则参数数量近似按

$$
N_{\rm par}\sim 2N_{\rm ant}N_{\rm dir}N_tN_\nu
$$

增长。方向数、时间分辨率和频率分辨率稍一增加，参数空间就会迅速膨胀。若没有足够强的先验约束，求解器很容易把噪声或天空模型误差拟合进去。

因此，3GC 往往需要正则化和物理先验。主波束应随频率平滑变化，电离层相位屏应在天空方向上具有空间相关性，增益解不应在相邻时间片无物理原因地剧烈跳变。把这些先验写入模型，和选择高效求解器同样重要。


![3GC 参数数量随方向和时间增长](figures/calibration_3gc_parameter_growth.png)

**图 8.4.4** 对 64 面天线的简化估算。方向数和时间解数量增加时，待求实参数数量快速增长，说明 3GC 必须依赖物理先验、正则化和高效成像/求解耦合。


### 8.4.6 什么时候需要从 2GC 升级到 3GC

完成 1GC 和 2GC 后，若残差仍明显随视场位置变化，图像边缘动态范围远差于相位中心，离轴亮源周围持续出现条纹或拉伸结构，同一源在不同时间、频率或偏振下呈现位置相关系统误差，就说明方向无关近似可能已经失效。此时继续缩短 2GC 解间隔、增加迭代次数或改动权重，通常只能局部缓解，无法改变误差的方向依赖本质。

3GC 的判断标准不是算法是否更高级，而是误差物理是否已经要求方向相关模型。一个良好的校准流程应在每一步都检查残差结构：若残差像全场噪声，方向无关校准可能已经足够；若残差围绕特定方向组织起来，就应回到 RIME，明确哪些 Jones 项必须带上方向依赖。
